# Data Cleaning 10 -- OptionMetrics Options Factors Panel

## Input
`Data/Data_Collection/Initial/10_OptionSuite/final/om_options_factors_panel.parquet` (999,844 rows, 2003--2025, 216 PERMNOs, ~34 factor columns)

## Purpose
Cleans the daily stock-level options factors panel, which was already merged from 5 sources during collection (OptionSuite base, VRP, vol surface, Greeks/positioning, borrow rates), deduplicated, and normalised (GEX/DEX divided by market cap, moneyness OI as % of total). Key concerns addressed: implied borrow rate sparsity, extreme values in PC_Ratio and normalised positioning factors, placeholder detection for OptionMetrics sentinel values, PERMNOs missing from OptionMetrics entirely, and date range trimming.

## Stage 0: Load & Inspect
- Loads panel and verifies PERMNO coverage against the master list
- Identifies 11 PERMNOs missing entirely from OptionMetrics (stocks with no listed options or options under different security identifiers)
- Reports shape, date range, column inventory

## Stage 1: Missing Data Audit
- Per-column NaN rates classified into tiers: 0%, <5%, 5--30%, >=30%
- Full listing of all factors sorted by NaN percentage

## Stage 2: Placeholder & Extreme Value Detection
- **Sentinel check:** verifies no -99.99 sentinel values remain (collection Stage 5 replaced these with NaN)
- **Extreme values (>10 sigma):** identifies factors with outlier values and reports their ranges
- **PC_Ratio deep dive:** detailed percentile analysis of the put-call ratio (max 291,812, but 99.9th percentile only 13.8). Counts values above 10, 100, and 1,000.
- **Negative value verification:** confirms that large negative values in `oi_wt_theta` (down to -2,295), `dex_norm` (-264), and `delta_dollar_volume_norm` are legitimate (theta is always negative for long options; dex_norm reflects real net short delta positions normalised by market cap), not placeholder sentinels.

## Stage 3: Per-Factor NaN by Year
- Year-by-year NaN heatmap for factors with >5% NaN, identifying temporal patterns
- Late-starting factor identification
- Early-stopping factor identification

## Stage 4: In-Universe NaN Rates
- Merges with `universe_annual` to restrict analysis to in-universe observations
- Per-factor NaN rates compared between in-universe and overall
- In-universe NaN tiers for kept factors

## Stage 5: Per-PERMNO NaN (In-Universe)
- Average NaN rate per PERMNO using only retained factors and in-universe observations
- Lists the 15 worst PERMNOs

## Stage 6: Duplicate & Coverage Checks
- Duplicate `(permno, date)` check
- Day-of-week distribution
- Stocks per day distribution

## Stage 8: Clean & Save

### Date Range Trimmed to 2004--2024
Raw data includes 2003 rows (pre-sample) and 2025 rows (out of scope). Trimmed to match the rest of the pipeline.

### Columns Dropped (3) -- Implied Borrow Rates
- `rate10` (77.9% NaN), `rate30` (52.3% NaN), `rate60` (42.4% NaN). OptionMetrics only computes borrow rates when put-call parity conditions are met, which fails for many stock-days. Too sparse to use.

### 11 PERMNOs Missing Entirely from OptionMetrics
12345, 14277, 45356, 65883, 69032, 79057, 79237, 80100, 85592, 89071, 92156 -- stocks with no listed options or options under different security identifiers. These will have NaN for all options factors in the merged dataset. Cross-sectional aggregation handles this (89+ of 100 stocks still contribute each day).

### No Forward-Fill
Stock-level daily data -- cross-sectional aggregation skips NaN naturally.

### No Winsorisation
PC_Ratio has extreme values (max 291,812, but 99.9th percentile is only 13.8). Winsorisation applied cross-sectionally per date in the merge pipeline.

### No Placeholder Replacement Needed
Values of <= -99 found in `oi_wt_theta`, `dex_norm`, and `delta_dollar_volume_norm` are legitimate financial values, not sentinels.

### In-Universe NaN Rates After Cleaning
14 factors at 0% NaN, 16 at <5%, only 1 above 5% (`sumOI_p_money3_pct` at 9.3%). Per-PERMNO: 207/216 below 5% average NaN, zero above 30%.

## Output
`Data/Data_Collection/Cleaned/10_OptionMetrics/om_options_factors_panel_clean.parquet` -- 31 factor columns (down from 34), date range 2004--2024

In [1]:
# %% [markdown]
# # Data Cleaning: om_options_factors_panel.parquet
#
# Source: Data/Data_Collection/Initial/10_OptionSuite/final/om_options_factors_panel.parquet
# Output: Data/Data_Collection/Cleaned/10_OptionMetrics/om_options_factors_panel_clean.parquet
#
# Daily stock-level options factors panel. Already merged from 5 sources during
# collection (OptionSuite base, VRP, vol surface, Greeks/positioning, borrow rates).
# Already filtered to universe PERMNOs, deduplicated, and normalised
# (GEX/DEX/OI divided by market cap, moneyness OI as % of total).
#
# Keyed on (permno, date). ~1M rows, ~35 factor columns.

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH    = Path('../../../Data/Data_Collection/Initial/10_OptionSuite/final/om_options_factors_panel.parquet')
MASTER_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_master_clean.parquet')
ANNUAL_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet')
OUT_DIR     = Path('../../../Data/Data_Collection/Cleaned/10_OptionMetrics')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — OptionMetrics Panel")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])
master = pd.read_parquet(MASTER_PATH)

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")
print(f"  Unique PERMNOs: {df['permno'].nunique()}")
print(f"  Master PERMNOs: {len(master)}")

# Verify PERMNOs
data_permnos = set(df['permno'].unique())
master_permnos = set(master['permno'])
extra = data_permnos - master_permnos
missing = master_permnos - data_permnos
print(f"\n  PERMNOs in data but NOT in master: {len(extra)}")
print(f"  PERMNOs in master but NOT in data: {len(missing)}")
if missing:
    print(f"    Missing ({len(missing)}): {sorted(list(missing))[:20]}...")

# Column inventory
factor_cols = [c for c in df.columns if c not in ['permno', 'date']]
print(f"\n  Factor columns ({len(factor_cols)}):")
for i, c in enumerate(factor_cols, 1):
    print(f"    {i:>3d}. {c:<30s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (5 rows) ---")
show_cols = ['permno', 'date'] + factor_cols[:8]
print(df[show_cols].head(5).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

# ── Per-column NaN ───────────────────────────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

# Tiers
tier_drop = col_nan_pct[col_nan_pct >= 30]
tier_mid = col_nan_pct[(col_nan_pct >= 5) & (col_nan_pct < 30)]
tier_clean = col_nan_pct[(col_nan_pct > 0) & (col_nan_pct < 5)]
tier_0 = col_nan_pct[col_nan_pct == 0]

print(f"\n  0% NaN:       {len(tier_0)}")
print(f"  <5% NaN:      {len(tier_clean)}")
print(f"  5-30% NaN:    {len(tier_mid)}")
print(f"  ≥30% NaN:     {len(tier_drop)}  ← DROP")

print(f"\n--- All factors sorted by NaN % ---")
print(f"\n  {'Factor':<35s} {'NaN %':>8s}  {'Count':>10s}")
print("  " + "-" * 55)
for col in col_nan_sorted.index:
    pct = col_nan_pct[col]
    count = int(col_nan[col])
    flag = " ← DROP" if pct >= 30 else ""
    print(f"  {col:<35s} {pct:>7.2f}%  {count:>10,d}{flag}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: PLACEHOLDER & EXTREME VALUE DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: PLACEHOLDER & EXTREME VALUE DETECTION")
print("=" * 90)

# ── Check for -99.99 sentinels (borrow rates had these) ─────────────────────
print(f"\n--- Checking for -99.99 sentinels ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    n = ((df[col] <= -99) | (df[col] == -99.99)).sum()
    if n > 0:
        print(f"  {col:<30s} {n:>6d} values ≤ -99")

print(f"  (Should be zero — collection Stage 5 replaced these with NaN)")

# ── Extreme values (>10σ) ────────────────────────────────────────────────────
print(f"\n--- Extreme values (>10 std from mean) ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) < 100:
        continue
    mean = vals.mean()
    std = vals.std()
    if std == 0:
        continue
    n_extreme = (((vals - mean).abs() / std) > 10).sum()
    if n_extreme > 0:
        extremes = vals[((vals - mean).abs() / std) > 10]
        print(f"  {col:<30s} {n_extreme:>6d} values >10σ  "
              f"(extreme range: [{extremes.min():.4f}, {extremes.max():.4f}])")

# ── PC_Ratio specifically (max 291,812 from diagnostics) ────────────────────
if 'PC_Ratio' in df.columns:
    print(f"\n--- PC_Ratio deep dive ---")
    vals = df['PC_Ratio'].dropna()
    print(f"  Range: [{vals.min():.4f}, {vals.max():.2f}]")
    print(f"  Mean: {vals.mean():.4f}, Median: {vals.median():.4f}")
    pctiles = vals.quantile([0.01, 0.05, 0.95, 0.99, 0.999, 0.9999])
    for p, v in pctiles.items():
        print(f"  {p*100:>6.2f}th percentile: {v:.4f}")
    n_gt10 = (vals > 10).sum()
    n_gt100 = (vals > 100).sum()
    n_gt1000 = (vals > 1000).sum()
    print(f"  PC_Ratio > 10:   {n_gt10:,} ({n_gt10/len(vals)*100:.2f}%)")
    print(f"  PC_Ratio > 100:  {n_gt100:,} ({n_gt100/len(vals)*100:.2f}%)")
    print(f"  PC_Ratio > 1000: {n_gt1000:,} ({n_gt1000/len(vals)*100:.2f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: PER-FACTOR NaN BY YEAR
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: PER-FACTOR NaN BY YEAR")
print("=" * 90)

df['_year'] = df['date'].dt.year
years = sorted(df['_year'].unique())

# Focus on factors with >5% NaN
notable_factors = list(col_nan_pct[col_nan_pct > 5].sort_values(ascending=False).index)

print(f"\n--- NaN % by year for factors with >5% NaN ---")
if len(notable_factors) > 0:
    nan_by_year = df.groupby('_year')[notable_factors].apply(
        lambda x: x.isna().mean() * 100
    ).round(1)

    header = f"  {'Factor':<30s}" + "".join(f" {y:>5d}" for y in years)
    print(f"\n{header}")
    print("  " + "-" * (30 + 6 * len(years)))

    for col in notable_factors:
        row = f"  {col:<30s}"
        for y in years:
            pct = nan_by_year.loc[y, col] if y in nan_by_year.index else 0
            if pct >= 90:
                row += f"  {'--':>5s}"
            elif pct >= 30:
                row += f" {pct:>4.0f}%"
            elif pct > 0:
                row += f" {pct:>4.1f}"
            else:
                row += f"  {'·':>5s}"
        print(row)

# ── Late-starting or early-stopping ─────────────────────────────────────────
print(f"\n--- Late-starting factors ---")
for col in notable_factors:
    yearly_nan = df.groupby('_year')[col].apply(lambda x: x.isna().mean() * 100)
    first_good = yearly_nan[yearly_nan < 50]
    if len(first_good) > 0 and first_good.index[0] > years[0]:
        print(f"  {col:<30s} starts ~{first_good.index[0]}")

print(f"\n--- Early-stopping factors ---")
for col in notable_factors:
    yearly_nan = df.groupby('_year')[col].apply(lambda x: x.isna().mean() * 100)
    last_good = yearly_nan[yearly_nan < 50]
    if len(last_good) > 0 and last_good.index[-1] < years[-1]:
        print(f"  {col:<30s} stops ~{last_good.index[-1]}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: IN-UNIVERSE NaN RATES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: IN-UNIVERSE NaN RATES")
print("=" * 90)

annual = pd.read_parquet(ANNUAL_PATH)

# Define kept factors (exclude ≥30% NaN)
drop_factors = list(tier_drop.index)
keep_factors = [c for c in factor_cols if c not in drop_factors]

print(f"\n  Factors kept: {len(keep_factors)}")
print(f"  Factors dropped: {len(drop_factors)}")
if drop_factors:
    print(f"  Dropped: {drop_factors}")

# Filter to in-universe
df_universe = df.merge(
    annual[['permno', 'year']],
    left_on=['permno', '_year'],
    right_on=['permno', 'year'],
    how='inner'
)
print(f"\n  All rows: {len(df):,}")
print(f"  In-universe rows: {len(df_universe):,}")
print(f"  Dropped (out of universe): {len(df) - len(df_universe):,}")

# Per-factor NaN in-universe
n_univ = len(df_universe)
univ_nan = df_universe[keep_factors].isna().sum()
univ_nan_pct = (univ_nan / n_univ * 100).round(2)
univ_nan_sorted = univ_nan_pct.sort_values(ascending=False)
overall_nan_pct = (df[keep_factors].isna().sum() / len(df) * 100).round(2)

print(f"\n--- Per-Factor NaN: In-Universe vs Overall (factors with >1% NaN) ---")
print(f"\n  {'Factor':<35s} {'Universe':>10s}  {'Overall':>10s}  {'Diff':>8s}")
print("  " + "-" * 68)
for col in univ_nan_sorted.index:
    u_pct = univ_nan_pct[col]
    o_pct = overall_nan_pct[col]
    diff = u_pct - o_pct
    if u_pct > 1:
        print(f"  {col:<35s} {u_pct:>9.2f}%  {o_pct:>9.2f}%  {diff:>+7.2f}%")

# In-universe NaN tiers
u_tier_0 = univ_nan_pct[univ_nan_pct == 0]
u_tier_clean = univ_nan_pct[(univ_nan_pct > 0) & (univ_nan_pct < 5)]
u_tier_mid = univ_nan_pct[(univ_nan_pct >= 5) & (univ_nan_pct < 15)]
u_tier_high = univ_nan_pct[(univ_nan_pct >= 15) & (univ_nan_pct < 30)]
u_tier_drop = univ_nan_pct[univ_nan_pct >= 30]

print(f"\n--- In-Universe NaN Tiers (kept factors) ---")
print(f"  0% NaN:       {len(u_tier_0)}")
print(f"  <5% NaN:      {len(u_tier_clean)}")
print(f"  5-15% NaN:    {len(u_tier_mid)}")
print(f"  15-30% NaN:   {len(u_tier_high)}")
print(f"  ≥30% NaN:     {len(u_tier_drop)}  ← should be zero!")

if len(u_tier_drop) > 0:
    print(f"\n  ⚠ Factors ≥30% NaN even in-universe:")
    for col in u_tier_drop.index:
        print(f"    {col:<35s} {u_tier_drop[col]:.2f}%")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5: PER-PERMNO NaN (IN-UNIVERSE)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 5: PER-PERMNO NaN (IN-UNIVERSE)")
print("=" * 90)

permno_nan_univ = (
    df_universe.groupby('permno')[keep_factors]
    .apply(lambda x: x.isna().mean().mean() * 100)
    .sort_values(ascending=False)
)

print(f"\n  PERMNOs with <5% avg NaN:    {(permno_nan_univ < 5).sum()}")
print(f"  PERMNOs with 5-15% avg NaN:  {((permno_nan_univ >= 5) & (permno_nan_univ < 15)).sum()}")
print(f"  PERMNOs with 15-30% avg NaN: {((permno_nan_univ >= 15) & (permno_nan_univ < 30)).sum()}")
print(f"  PERMNOs with >30% avg NaN:   {(permno_nan_univ >= 30).sum()}")

worst = permno_nan_univ.head(15)
print(f"\n  15 worst PERMNOs (in-universe, kept factors):")
print(f"  {'PERMNO':>8s}  {'Avg NaN %':>10s}  {'Univ Rows':>10s}  {'Yrs':>5s}")
print("  " + "-" * 40)
for permno, pct in worst.items():
    n_rows_u = len(df_universe[df_universe['permno'] == permno])
    n_yrs = len(annual[annual['permno'] == permno])
    print(f"  {int(permno):>8d}  {pct:>9.2f}%  {n_rows_u:>10,d}  {n_yrs:>5d}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 6: DUPLICATE & COVERAGE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 6: DUPLICATE & COVERAGE CHECKS")
print("=" * 90)

# Duplicates
n_dupes = df.duplicated(subset=['permno', 'date']).sum()
print(f"\n  Duplicate (permno, date): {n_dupes}")

# Day-of-week
print(f"\n--- Day-of-week distribution ---")
dow = df['date'].dt.day_name().value_counts()
print(dow.to_string())

# Stocks per day
stocks_per_day = df.groupby('date')['permno'].nunique()
print(f"\n--- Stocks per day ---")
print(f"  Mean: {stocks_per_day.mean():.1f}")
print(f"  Min:  {stocks_per_day.min()} (on {stocks_per_day.idxmin().date()})")
print(f"  Max:  {stocks_per_day.max()} (on {stocks_per_day.idxmax().date()})")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 7: SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 7: SUMMARY")
print("=" * 90)

print(f"""
FACTOR TIERS (overall):
  0% NaN:       {len(tier_0):>4d}
  <5% NaN:      {len(tier_clean):>4d}
  5-30% NaN:    {len(tier_mid):>4d}
  ≥30% NaN:     {len(tier_drop):>4d}  ← DROP
  ────────────────────────
  TOTAL KEEP:   {len(keep_factors):>4d}
  TOTAL DROP:   {len(drop_factors):>4d}

Paste back the output and I will write the cleaning cell.
""")

# Clean up
df = df.drop(columns=['_year'], errors='ignore')

STAGE 0: LOAD & INSPECT — OptionMetrics Panel

  Shape: 999,844 rows × 36 columns
  Date range: 2003-01-02 → 2025-08-29
  Unique dates: 5,702
  Unique PERMNOs: 216
  Master PERMNOs: 227

  PERMNOs in data but NOT in master: 0
  PERMNOs in master but NOT in data: 11
    Missing (11): [np.int64(12345), np.int64(14277), np.int64(45356), np.int64(65883), np.int64(69032), np.int64(79057), np.int64(79237), np.int64(80100), np.int64(85592), np.int64(89071), np.int64(92156)]...

  Factor columns (34):
      1. iv_catm                        float64        
      2. iv_PATM                        float64        
      3. iv_POTM                        float64        
      4. Skew_OTM                       float64        
      5. Parity_VSpread                 float64        
      6. nopt_Parity                    float64        
      7. PC_Ratio                       float64        
      8. hvol                           float64        
      9. rv_30d                         float64      

In [2]:
# %% [markdown]
# ## Stage 8: Clean & Save
#
# **Data overview:**
# Daily stock-level options factors panel from OptionMetrics. 999,844 rows
# across 2003–2025 for 216 PERMNOs. Already merged from 5 sources during
# collection (OptionSuite base, VRP, vol surface, Greeks/positioning, borrow
# rates), deduplicated, and normalised (GEX/DEX divided by market cap,
# moneyness OI as % of total).
#
# **Date range trimmed to 2004–2024:**
# Raw data includes 2003 rows (pre-sample) and 2025 rows (out of scope).
# Trimmed to match the rest of the pipeline.
#
# **Columns dropped (3) — implied borrow rates:**
# - `rate10` (77.9% NaN), `rate30` (52.3% NaN), `rate60` (42.4% NaN).
#   OptionMetrics only computes borrow rates when put-call parity conditions
#   are met, which fails for many stock-days. Too sparse to use.
#
# **11 PERMNOs missing entirely from OptionMetrics:**
# 12345, 14277, 45356, 65883, 69032, 79057, 79237, 80100, 85592, 89071,
# 92156 — stocks with no listed options or options under different security
# identifiers. These will have NaN for all options factors in the merged
# dataset. Cross-sectional aggregation handles this (89+ of 100 stocks
# still contribute each day).
#
# **No forward-fill.** Stock-level daily data — cross-sectional aggregation
# skips NaN naturally.
#
# **No winsorisation.** PC_Ratio has extreme values (max 291,812, but 99.9th
# percentile is only 13.8). Winsorisation applied cross-sectionally per date
# in the merge pipeline.
#
# **No placeholder replacement needed.** The ≤ -99 values found in
# `oi_wt_theta` (10,402), `dex_norm` (304), and `delta_dollar_volume_norm` (2)
# are legitimate: theta is always negative and can reach -2,295 for
# near-expiry heavy portfolios; dex_norm of -264 is a real large net short
# delta position normalised by market cap. Not sentinels.
#
# **In-universe NaN rates are excellent after dropping borrow rates:**
# 14 factors at 0% NaN, 16 at <5%, only 1 above 5% (sumOI_p_money3_pct
# at 9.3%). Per-PERMNO: 207/216 below 5% average NaN, zero above 30%.
#
# **Factors retained: 31** (was 34 before cleaning)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 8: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 8: CLEAN & SAVE")
print("=" * 90)

# ── 8a. Trim date range to 2004–2024 ────────────────────────────────────────
n_before = len(df)
df = df[(df['date'] >= '2004-01-01') & (df['date'] <= '2024-12-31')].reset_index(drop=True)
n_trimmed = n_before - len(df)
print(f"\n  Trimmed date range to 2004–2024: dropped {n_trimmed:,} rows")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# ── 8b. Drop borrow rate columns ────────────────────────────────────────────
drop_cols = ['rate10', 'rate30', 'rate60']
drop_cols_present = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols_present)

factor_cols_final = [c for c in df.columns if c not in ['permno', 'date']]
print(f"\n  Dropped {len(drop_cols_present)} columns: {drop_cols_present}")
print(f"  Remaining factor columns: {len(factor_cols_final)}")

# ── 8c. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols_final].isna().sum()
nan_cols = nan_check[nan_check > 0].sort_values(ascending=False)
total_nan = nan_cols.sum()
total_cells = len(df) * len(factor_cols_final)

print(f"\n  Total NaN: {total_nan:,} / {total_cells:,} ({total_nan/total_cells*100:.2f}%)")
print(f"  Factors with any NaN: {len(nan_cols)} / {len(factor_cols_final)}")

if len(nan_cols) > 0:
    print(f"\n  NaN per factor:")
    print(f"  {'Factor':<35s} {'NaN':>8s}  {'%':>7s}")
    print("  " + "-" * 55)
    for col in nan_cols.index:
        n = int(nan_cols[col])
        pct = n / len(df) * 100
        print(f"  {col:<35s} {n:>8,d}  {pct:>6.2f}%")

# ── 8d. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  PERMNOs: {df['permno'].nunique()}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols_final)} factors):")
for i, c in enumerate(factor_cols_final, 1):
    vals = df[c].dropna()
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n:,} NaN, {nan_n/len(df)*100:.1f}%)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<30s} [{vals.min():.4f}, {vals.max():.4f}]{nan_str}")

print(f"\n  Sample (first 5 rows):")
show_cols = ['permno', 'date'] + factor_cols_final[:8]
print(df[show_cols].head(5).to_string(index=False))

# ── 8e. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'om_options_factors_panel_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns ({len(factor_cols_final)} factors)")

print("\nCleaning complete.")

STAGE 8: CLEAN & SAVE

  Trimmed date range to 2004–2024: dropped 69,089 rows
  Date range: 2004-01-02 → 2024-12-31

  Dropped 3 columns: ['rate10', 'rate30', 'rate60']
  Remaining factor columns: 31

  Total NaN: 486,695 / 28,853,405 (1.69%)
  Factors with any NaN: 31 / 31

  NaN per factor:
  Factor                                   NaN        %
  -------------------------------------------------------
  sumOI_p_money3_pct                   126,721   13.61%
  sumOI_c_money3_pct                    70,052    7.53%
  sumOI_c_money1_pct                    38,221    4.11%
  sumOI_p_money1_pct                    35,036    3.76%
  Skew_OTM                              29,599    3.18%
  iv_POTM                               29,240    3.14%
  vrp_hvol                              13,293    1.43%
  vrp_rv                                13,217    1.42%
  rv_30d                                12,540    1.35%
  total_volume_norm                      8,744    0.94%
  total_oi_norm                 